# LC 42 — Trapping Rain Water

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Every column can trap water up to
<code>min(max_left, max_right) - height[i]</code>. The two-pointer
trick lets us compute this in one pass: whichever side has the
smaller running max is the binding constraint — process that side.
</div>

## Official Problem Statement

Given `n` non-negative integers representing an elevation map where
the width of each bar is 1, compute how much water it can trap
after raining.

**Example 1:**
```
Input:  height = [0,1,0,2,1,0,1,3,2,1,2,1]
Output: 6
```
**Example 2:**
```
Input:  height = [4,2,0,3,2,5]
Output: 9
```
**Constraints:**
- `n == height.length`
- `1 <= n <= 2 * 10^4`
- `0 <= height[i] <= 10^5`

## What This Is Actually Asking

Imagine pouring water over a skyline of bars — water collects in
the valleys between taller bars. For each bar, the water level
above it equals the shorter of the tallest bar to its left and the
tallest bar to its right, minus the bar's own height.

A naive approach precomputes left-max and right-max arrays (O(n)
space). The two-pointer approach eliminates that extra space by
observing: if lmax < rmax, the left bar's water is determined
entirely by lmax, regardless of what's on the right.

We move inward from both ends, always processing the side with
the smaller max, accumulating water along the way.

## Walk Through an Example by Hand

```
height = [4, 2, 0, 3, 2, 5]
left=0  right=5  lmax=0  rmax=0  water=0

Step 1: h[0]=4, h[5]=5
  lmax=max(0,4)=4, rmax=max(0,5)=5
  lmax(4) < rmax(5) → process left
  water += max(0, 4-4) = 0   left→1

Step 2: h[1]=2, lmax=4, rmax=5
  lmax < rmax → process left
  water += max(0, 4-2) = 2   left→2

Step 3: h[2]=0, lmax=4, rmax=5
  lmax < rmax → process left
  water += max(0, 4-0) = 4   left→3

Step 4: h[3]=3, lmax=4, rmax=5
  lmax < rmax → process left
  water += max(0, 4-3) = 1   left→4

Step 5: h[4]=2, lmax=4, rmax=5
  lmax < rmax → process left
  water += max(0, 4-2) = 2   left→5

left == right, done.  water = 0+2+4+1+2 = 9 ✓
```

## The Picture

```
height = [4, 2, 0, 3, 2, 5]

5 |              █
4 |█  ~  ~  ~  ~ █    ~ = water
3 |█  ~  ~  █  ~ █
2 |█  █  ~  █  █ █
1 |█  █  ~  █  █ █
0 |█  █  █  █  █ █
   0  1  2  3  4  5

Two-pointer shrinks the window:

  lmax tracks tallest seen from left
  rmax tracks tallest seen from right

  [L -----> ... <----- R]
   lmax?         rmax?

  If lmax < rmax: left side is safe to process
    water at L = lmax - h[L]  (if positive)
    advance L right
  Else: right side is safe to process
    water at R = rmax - h[R]  (if positive)
    advance R left
```

## When To Use This Pattern

- When the answer at index `i` depends on a global property
  (max left, max right), think **prefix/suffix precomputation**.
- When two prefix scans can be collapsed by choosing the binding
  constraint side, think **two pointers from both ends**.
- When you see "elevation map" or "container" problems, think
  **two-pointer + running max**.
- When array traversal needs O(1) extra space, think
  **converging two pointers** instead of auxiliary arrays.
- When constraints guarantee non-negative contribution (water
  can't be negative), think **accumulate in place**.

## The Approach

Place two pointers at both ends. Maintain running maximums `lmax`
and `rmax`. At each step, the side with the smaller max is
the bottleneck — its contribution is fully determined, so process
it: add `max(0, running_max - height[ptr])` to water, update the
running max, and advance the pointer inward.

This works because if `lmax < rmax`, the water at the left pointer
is `lmax - h[left]` regardless of what lies between — the right
wall is tall enough to guarantee that level. Continue until the
pointers meet.

In [1]:
# Imports
from typing import List

In [2]:
# ----------------------------------------------------------
# Harness
# ----------------------------------------------------------
def test_harness(func):
    cases = [
        ([0,1,0,2,1,0,1,3,2,1,2,1], 6),
        ([4,2,0,3,2,5],             9),
        ([3,0,2,0,4],               7),
        ([1,0,1],                   1),
        ([0],                       0),
        ([4,2,3],                   1),
    ]
    passed = 0
    for height, expected in cases:
        result = func(height)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if not ok:
            print(f"{status} | input={height}"
                  f" | expected={expected}"
                  f" | got={result}")
        else:
            print(f"{status} | input={height}"
                  f" | water={result}")
        passed += ok
    print(f"\n{passed}/{len(cases)} tests passed")

In [6]:
def trap(height: List[int]) -> int:
    """
    LC 42 — Trapping Rain Water

    Two-pointer approach:
    - left, right pointers from both ends
    - lmax, rmax track running max from each side
    - Process the side with the smaller max:
        water += max(0, running_max - h[ptr])
    - Continue until pointers meet

    Time:  O(n)
    Space: O(1)
    """
    # main concept . water requires bondaries both sides to be trapped
    # First and Last Index can never trap water
    # if the left max <height> is shorter than the right; 
    # you can trap water as much as the left side max - current height
    if not height:return 0            # edge case
    res , l , r = 0, 0, len(height) - 1
    lMax, rMax = height[l], height[r]
    while l < r:
        if lMax < rMax:
            l += 1
            lMax = max( lMax, height[l])
            res += lMax - height[l]
        else:
            r -= 1
            rMax = max( rMax, height[r])
            res +=  rMax - height[r]
    return res
'''r
PASSED | input=[0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1] | water=6
PASSED | input=[4, 2, 0, 3, 2, 5] | water=9
PASSED | input=[3, 0, 2, 0, 4] | water=7
PASSED | input=[1, 0, 1] | water=1
PASSED | input=[0] | water=0
PASSED | input=[4, 2, 3] | water=1

6/6 tests passed
'''
test_harness(trap)            


PASSED | input=[0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1] | water=6
PASSED | input=[4, 2, 0, 3, 2, 5] | water=9
PASSED | input=[3, 0, 2, 0, 4] | water=7
PASSED | input=[1, 0, 1] | water=1
PASSED | input=[0] | water=0
PASSED | input=[4, 2, 3] | water=1

6/6 tests passed


In [ ]:
# Uncomment and run when solution is ready
# test_harness(trap)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute Force (nested) | O(n²) | O(1) | For each bar scan L & R |
| Prefix/Suffix arrays | O(n) | O(n) | Two extra arrays |
| Two Pointers (optimal) | O(n) | O(1) | Single pass, no arrays |
| Monotonic stack | O(n) | O(n) | Horizontal water slabs |

## Real World Connection

**AWS / Data Engineering context:** This pattern models capacity
planning under bilateral constraints — think of storage buffers
between pipeline stages where throughput is bottlenecked by the
slowest adjacent stage.

At Citi, risk aggregation across desks is similar: each desk's
net exposure is bounded by the minimum of its upstream and
downstream limits, and you scan the portfolio to find trapped
liquidity exactly as this algorithm finds trapped water.

In streaming systems (Kinesis, Kafka), the two-pointer intuition
maps to back-pressure: you always process the side whose
constraint is already determined, rather than waiting for full
global information.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra